# Pandas — l'agrégation avec `groupby` (cours)

Cellules d'exemple à **exécuter** (contexte : table de salariés). Les exercices (contexte : essais cliniques) sont tout en bas.

### Le df d'exemple

In [1]:
import pandas as pd
df = pd.DataFrame({
    "nom": ["Alice", "Bob", "Chloé", "David"],
    "salaire": [38000, 41000, 45000, 39000],
    "service": ["Data", "Data", "RH", "RH"],
    "site": ["Rouen", "Paris", "Rouen", "Paris"],
})
df

,nom,salaire,service,site
0,Alice,38000,Data,Rouen
1,Bob,41000,Data,Paris
2,Chloé,45000,RH,Rouen
3,David,39000,RH,Paris


### groupby + une agrégation

In [2]:
print(df.groupby("service")["salaire"].mean())   # salaire moyen par service
print(df.groupby("service").size())              # nb de lignes par groupe

service
Data    39500.0
RH      42000.0
Name: salaire, dtype: float64
service
Data    2
RH      2
dtype: int64


### Plusieurs calculs : .agg()

In [3]:
df.groupby("service")["salaire"].agg(["mean", "min", "max", "count"])

,mean,min,max,count
service,,,,
Data,39500.0,38000,41000,2
RH,42000.0,39000,45000,2


In [4]:
# named aggregation : des noms de colonnes clairs
df.groupby("service").agg(
    salaire_moyen = ("salaire", "mean"),
    effectif      = ("salaire", "count"),
)

,salaire_moyen,effectif
service,,
Data,39500.0,2
RH,42000.0,2


### Grouper par plusieurs colonnes

In [5]:
df.groupby(["service", "site"])["salaire"].mean().reset_index()

,service,site,salaire
0,Data,Paris,41000.0
1,Data,Rouen,38000.0
2,RH,Paris,39000.0
3,RH,Rouen,45000.0


# Exercices

Dataset : `data/essais_cliniques.csv` (essais cliniques d'un labo : phase, aire thérapeutique, site, patients recrutés, budget, statut).
**À toi de choisir tes outils — rien n'est imposé, et méfie-toi de ce qui manque dans les données.**

1. Combien de patients recrutés **au total par aire thérapeutique** ? Laquelle recrute le plus ?
2. Quel est le **budget moyen par phase** ? Commente l'ordre de grandeur.
3. Quels **sites** portent le plus d'essais ? (classement)
4. Pour **chaque aire**, d'un seul coup : nombre d'essais, total patients, budget moyen (tableau lisible).
5. Budget moyen par **(aire thérapeutique, statut)**. Des combinaisons ressortent-elles ?
6. Une colonne a des valeurs manquantes : **repère-la**, dis (en commentaire) si elle fausse un calcul précédent, et corrige si besoin.
7. Réflexion (pas de code) : que fait `groupby` d'une ligne dont la colonne de regroupement est NaN ?

In [37]:
import pandas as pd 
df = pd.read_csv("data\\essais_cliniques.csv")
df

,essai_id,phase,aire_therapeutique,site,patients_recrutes,budget,statut
0,ESS001,Phase I,Neurologie,Rouen,11,225212.0,Recrutement
1,ESS002,Phase III,Neurologie,Paris,117,1815617.0,Recrutement
2,ESS003,Phase I,Oncologie,Paris,35,222413.0,Recrutement
3,ESS004,Phase III,Cardiologie,Toulouse,303,NaN,En cours
4,ESS005,Phase I,Cardiologie,Toulouse,26,298019.0,En cours
5,ESS006,Phase II,Diabétologie,Toulouse,31,617711.0,Recrutement
6,ESS007,Phase II,Cardiologie,Paris,99,NaN,Recrutement
7,ESS008,Phase II,Cardiologie,Toulouse,107,735922.0,Terminé
8,ESS009,Phase II,Immunologie,Lille,109,660669.0,En cours
9,ESS010,Phase III,Oncologie,Lille,261,3281491.0,Recrutement


In [20]:
## 1
df_patient = df.groupby("aire_therapeutique")["patients_recrutes"].sum()
df_patient

aire_therapeutique
Cardiologie     1885
Diabétologie     757
Immunologie      619
Neurologie      1415
Oncologie       1147
Name: patients_recrutes, dtype: int64

In [ ]:
## 2
df_phase = df.groupby("phase")["budget"].mean()
df_phase

# On remarque que par phase on obtient budget d'un ordre de grandeur entre 10^5 et 10^6 ce qui est énorme (n'y connaissant rien en budget ou en CA pour la pharma je ne me rends si c'est énorme mais je suppose que oui)  
# A part pour les  phase 2 qui coûte en moyenne un peu moins d'1 millions d'euros les autres phases sont plus proches des 3 millions  

phase
Phase I      2.898802e+05
Phase II     8.077247e+05
Phase III    2.560858e+06
Name: budget, dtype: float64

In [25]:
## 3
df_essai = df.groupby("site")["essai_id"].count()
df_essai

site
Lille       10
Lyon        11
Paris       11
Rouen        7
Toulouse    13
Name: essai_id, dtype: int64

In [28]:
## 4
df_agg_aire = df.groupby("aire_therapeutique").agg(
    nb_essai = ("essai_id", "count"),
    total_patient = ("patients_recrutes", "sum"),
    budget_moyen = ("budget", "mean"),
)

df_agg_aire

,nb_essai,total_patient,budget_moyen
aire_therapeutique,,,
Cardiologie,14,1885,1.198076e+06
Diabétologie,8,757,1.235962e+06
Immunologie,8,619,1.108708e+06
Neurologie,13,1415,1.294142e+06
Oncologie,9,1147,1.373496e+06


In [36]:
## 5
df_budget_2 = df.groupby(["aire_therapeutique", "statut"])["budget"].mean().reset_index()
df_budget_2

# combinaison service + tous les statuts (en cours, recrutement, terminé) + pour chaque statut de chaque aire thérapeutique, le budget moyen 

,aire_therapeutique,statut,budget
0,Cardiologie,En cours,1.053524e+06
1,Cardiologie,Recrutement,9.391800e+05
2,Cardiologie,Terminé,1.746076e+06
3,Diabétologie,En cours,3.205820e+05
4,Diabétologie,Recrutement,9.043167e+05
5,Diabétologie,Terminé,3.146278e+06
6,Immunologie,En cours,4.435878e+05
7,Immunologie,Recrutement,3.476317e+06
8,Immunologie,Terminé,1.206333e+06
9,Neurologie,En cours,1.707852e+06


In [ ]:
## 6
df

# La colonne budget possède des NaN, techniquement pour notre exercice ce n'est pas dérangeant car avec les aggrégations les NaN disparaissent 
# Cependant, on a fait des calculs sur le budget et il nous manque donc des valeurs donc on peut écrire 

# df = df.fillna(0)  ## Pour remplacer les NaN par 0 dans df  

# Et comme ça on garde les valeurs et ces dernières seront prises en compte si on fait des aggrégations sur le budget

,essai_id,phase,aire_therapeutique,site,patients_recrutes,budget,statut
0,ESS001,Phase I,Neurologie,Rouen,11,225212.0,Recrutement
1,ESS002,Phase III,Neurologie,Paris,117,1815617.0,Recrutement
2,ESS003,Phase I,Oncologie,Paris,35,222413.0,Recrutement
3,ESS004,Phase III,Cardiologie,Toulouse,303,0.0,En cours
4,ESS005,Phase I,Cardiologie,Toulouse,26,298019.0,En cours
5,ESS006,Phase II,Diabétologie,Toulouse,31,617711.0,Recrutement
6,ESS007,Phase II,Cardiologie,Paris,99,0.0,Recrutement
7,ESS008,Phase II,Cardiologie,Toulouse,107,735922.0,Terminé
8,ESS009,Phase II,Immunologie,Lille,109,660669.0,En cours
9,ESS010,Phase III,Oncologie,Lille,261,3281491.0,Recrutement


In [35]:
## 7 (réflexion en commentaire)
# Elle ne prend pas en compte la ligne c'est çà dire que si on a 12 NaN sur un total de 200 valeurs, alors groupby fera l'aggégation sur 188 valeurs 